# CNN'lerde Nümerik Optimizasyon Algoritmalarının Yakınsama ve Kararlılık Analizi

**MAT353 – Nümerik Analiz · 2026–2027 Güz · İstanbul Medeniyet Üniversitesi**

**Ekip:** _Ad Soyad (Öğrenci No)_, _Ad Soyad (Öğrenci No)_

SGD, Momentum, RMSprop ve Adam optimizasyon algoritmaları formüllerinden sıfırdan yazılmış (`optimizers.py`) ve sığ (3 konvolüsyon) ile BatchNorm'suz derin (8 konvolüsyon) CNN'lerde Fashion-MNIST üzerinde karşılaştırılmıştır.

| Deney | Soru | Çıktı |
|---|---|---|
| 1 | Yakınsama hızı ve seed kararlılığı (5 seed, ortalama ± std) | `exp1_summary.csv`, `loss_convergence.png`, `gradient_norm.png`, `time_vs_acc.png` |
| 2 | Öğrenme oranı duyarlılığı ve nümerik patlama (NaN/Inf) | `exp2_lr_stability.csv`, `lr_heatmap.png` |
| 3 | Gradyan doğrulaması: sonlu fark kesme/yuvarlama hatası | `exp3_fd_error.csv`, `fd_error.png` |
| 4 | CPU – GPU performans karşılaştırması | `exp4_cpu_gpu.csv`, `cpu_gpu.png` |

**Tekrarlanabilirlik:** seed'ler 42, 100, 2024, 7, 99; `random`, `numpy`, `torch` seed'leri ve `cudnn.deterministic=True` sabitlenmiştir.

> Çalıştırma: *Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU*, ardından *Tümünü çalıştır*.

In [ ]:
# === Ayarlar ===
RUN_FULL = False   # True: Deney 1-2'yi baştan çalıştırır (L4 GPU'da yaklaşık 1 saat)
USE_DRIVE = True   # Sonuçları Google Drive'da kalıcı tut
REPO = "https://github.com/cyberperfo/numericanaliyssproject.git"

In [ ]:
# === Proje dosyalarını al ===
import os

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive'
else:
    BASE = '/content'

PROJ = f'{BASE}/numericanaliyssproject'
if os.path.exists(PROJ):
    !git -C {PROJ} pull
else:
    !git clone {REPO} {PROJ}
%cd {PROJ}
!ls

In [ ]:
# === Ortam kontrolü ===
import torch, numpy, scipy, pandas, matplotlib
print('PyTorch', torch.__version__, '| NumPy', numpy.__version__, '| SciPy', scipy.__version__,
      '| pandas', pandas.__version__, '| Matplotlib', matplotlib.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK (CPU ile çalışılacak)')

## 1. Modeller
`ShallowCNN`: 3 konvolüsyon + 2 tam bağlantılı katman. `DeepCNN`: 8 konvolüsyon + 2 tam bağlantılı katman, **BatchNorm yok** (gradyan sönmesi/patlamasını gözlemlemek için).

In [ ]:
!python models.py

## 2. Test süreçleri (birim testleri)
Optimizer formüllerinin elle hesaplanan değerlerle ve PyTorch referansıyla eşleşmesi, determinizm, NaN/Inf tespiti, gradyan normu ölçümü ve sonlu fark doğrulaması test edilir (~1 dk).

In [ ]:
!python -m unittest tests -v

## 3. Canlı demo (~2 dk)
Küçük veri alt kümesiyle (6000 eğitim / 2000 test örneği, 2 epoch, 2 seed) tüm hattın çalıştığını gösterir. Sonuçlar `results_demo/` ve `plots_demo/` klasörlerine yazılır, asıl sonuçlara dokunmaz.

In [ ]:
!python experiments.py --quick --exp 1 2 --results-dir results_demo --plots-dir plots_demo

from IPython.display import Image, display
display(Image('plots_demo/loss_convergence.png', width=850))

## 4. Tam deneyler
Deney 1 (2 model × 4 optimizer × 5 seed × 10 epoch) ve Deney 2 (DeepCNN, 4 optimizer × 5 öğrenme oranı). Drive'da kayıtlı sonuç varsa ve `RUN_FULL = False` ise yeniden eğitim yapılmaz.

In [ ]:
have_12 = all(os.path.exists(f'results/{f}') for f in
              ['exp1_runs.csv', 'exp1_epochs.csv', 'exp2_lr_stability.csv'])
if RUN_FULL or not have_12:
    !python experiments.py --exp 1 2 --no-plot
else:
    print('Kayıtlı Deney 1-2 sonuçları kullanılıyor. Yeniden çalıştırmak için RUN_FULL = True yapın.')
    !python experiments.py --resummarize

## 5. Gradyan doğrulaması ve CPU–GPU karşılaştırması (~1-2 dk)
Deney 3: merkezi sonlu fark $D_h = \frac{L(\theta + h v) - L(\theta - h v)}{2h}$ ile otomatik türev $\nabla L \cdot v$ karşılaştırılır. Toplam hata $\approx \frac{h^2}{6}|L'''| + \frac{\varepsilon_{mach}|L|}{h}$; en iyi adım $h^* \sim \varepsilon_{mach}^{1/3}$.

Deney 4: iterasyon başına süre (ileri + geri yayılım + Adam adımı).

In [ ]:
!python experiments.py --exp 3 4 --no-plot

## 6. Sonuç tabloları

In [ ]:
import pandas as pd
pd.set_option('display.width', 250, 'display.max_columns', 30)

print('Deney 1 – Yakınsama ve seed kararlılığı (ortalama ± std, 5 seed)')
display(pd.read_csv('results/exp1_summary.csv'))
print('Deney 2 – Öğrenme oranı kararlılığı (DeepCNN)')
display(pd.read_csv('results/exp2_lr_stability.csv'))
print('Deney 4 – CPU / GPU')
display(pd.read_csv('results/exp4_cpu_gpu.csv'))

## 7. Grafikler (300 DPI, `plots/`)

In [ ]:
!python plot_results.py

from IPython.display import Image, Markdown, display
captions = {
    'loss_convergence': 'Şekil 1 – Eğitim kaybı (log ölçek), 5 seed ortalaması ± std',
    'gradient_norm':    'Şekil 2 – Katman bazlı gradyan normları ve ilk/son katman oranı',
    'lr_heatmap':       'Şekil 3 – Öğrenme oranı – optimizer kararlılık haritası (DeepCNN)',
    'time_vs_acc':      'Şekil 4 – Eğitim süresi – test doğruluğu',
    'fd_error':         'Şekil 5 – Sonlu fark göreli hatası (kesme / yuvarlama dengesi)',
    'cpu_gpu':          'Şekil 6 – CPU / GPU iterasyon süresi',
}
for name, cap in captions.items():
    path = f'plots/{name}.png'
    if os.path.exists(path):
        display(Markdown(f'**{cap}**'))
        display(Image(path, width=900))